In [1]:
import torch
import pickle
from torch.nn.utils.rnn import pad_sequence

# ロード
dev_data=pickle.load(open('71_sst_data.pkl','rb'))['dev']
model = BoWClassifier(np.load('70_embeddings.npy'))
model.load_state_dict(torch.load('73_bow_trained.pth'))
model.eval()

def make_batch(batch):
    x=pad_sequence([ex['input_ids'] for ex in batch], batch_first=True, padding_value=0)
    y=torch.cat([ex['label'] for ex in batch]).squeeze()
    return x,y

# 評価
correct=0
for i in range(0,len(dev_data),32):
    batch=dev_data[i:i+32]
    x,y=make_batch(batch)
    pred=model(x).round()
    correct+= (pred==y).sum().item()
acc=correct/len(dev_data)
print(f"Dev Accuracy: {acc:.4f}")

NameError: name 'BoWClassifier' is not defined

In [2]:
# knock74.ipynb
import numpy as np
import torch
import torch.nn as nn
import pickle
from torch.nn.utils.rnn import pad_sequence

# 70 & 71 のデータロード
embedding_matrix = np.load('70_embeddings.npy')
with open('71_sst_data.pkl', 'rb') as f:
    dev_data = pickle.load(f)['dev']

# バッチ作成
def make_batch(batch):
    x = pad_sequence([ex['input_ids'] for ex in batch], batch_first=True, padding_value=0)
    y = torch.cat([ex['label'] for ex in batch]).squeeze()
    return x, y

# モデル定義を再現
class BoWClassifier(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float), freeze=True, padding_idx=0)
        emb_dim = embedding_matrix.shape[1]
        self.weight = nn.Parameter(torch.zeros(emb_dim))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        mask = (input_ids != 0).unsqueeze(-1).float()
        summed = (emb * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)
        avg = summed / lengths
        logits = avg.matmul(self.weight) + self.bias
        return torch.sigmoid(logits)

if __name__ == '__main__':
    model = BoWClassifier(embedding_matrix)
    model.load_state_dict(torch.load('73_bow_trained.pth'))
    model.eval()

    correct = 0
    for i in range(0, len(dev_data), 32):
        batch = dev_data[i:i+32]
        x, y = make_batch(batch)
        pred = model(x).round()
        correct += (pred == y).sum().item()

    acc = correct / len(dev_data)
    print(f"Dev Accuracy: {acc:.4f}")

Dev Accuracy: 0.7856
